GRPO 会让模型推理变长，核心原因是：
- <font color='red'>它把“整条回答是否正确”的标量奖励，广播到每一个 token 上；</font>
- <font color='red'>同时损失函数里又按响应长度做归一化。 </font>

这两个机制叠加后，模型会发现：
- 在答错时，多写一点 token 能稀释惩罚；
- 在答对时，多写一点 token 也可能更容易维持高奖励。 

<font color='blue'>于是长度会自发膨胀。</font>

#  长度归一化带来的优化偏差

GRPO 的组内优势通常是：
$A_i = \frac{R(q,o_i) - mean(R)}{std(R)}$

 
但在很多实现里，token 级损失还会再除以响应长度，比如用 masked_sum / mask.sum() 做平均。
这就产生了一个隐藏的长度偏置：

| 情况 | 梯度效应 | 模型学到的倾向 |
|---|---|---|
| 回答正确，优势为正 | <font color='blue'>短回答</font>的单位长度梯度更大 | 倾向于生成更短的正确答案 |
| 回答错误，优势为负 | <font color='blue'>长回答</font>的单位长度惩罚更小 | 倾向于生成更长的错误答案 |
| 训练中有大量难题 | 错误样本多、长回答多 | 整体长度被持续推高 |

所以 GRPO 并不是单纯“鼓励长推理”，而是：<font color='red'>错误样本里，长回答更不容易被重罚</font>。 模型为了规避惩罚，会慢慢学会多写、多验证、多自我反思。

# 单标量奖励广播到所有 token
GRPO 的奖励通常是序列级的：整条回答对或错，得到一个标量。<font color='blue'>但这个标量会被广播到这条回答里的每个 token。</font>

<font color='red'>这意味着模型很难区分：</font>
  - 哪些 token 是真正有效的推理；
  - 哪些只是重复验证、冗余解释、无意义兜圈；
  - 哪些 token 只是让回答“看起来更完整”。
  
- 只要最终答案正确，长回答里的所有 token 都会获得正优势；
- 如果最终答案错误，长回答里的惩罚又被长度归一化稀释。于是模型会偏向“多写一点更保险”。

# 长回答更容易碰对，形成正反馈

在复杂推理任务里，更长的思维链往往更容易覆盖正确步骤。即便很多步骤是冗余的，只要最终答案对了，整条轨迹就会获得高奖励。

这会让模型学到一种策略：
与其冒险短答，不如多展开几步，增加命中正确答案的概率。
于是训练过程中会出现典型的“长度膨胀”：奖励提升放缓后，输出长度仍然继续增长。

# 这也解释了为什么会出现“<font color='red'>又长又错</font>”

如果只是单纯学会长链推理，模型应该越长越准。但 GRPO 训练出的模型有时会出现“又长又啰嗦、最后仍然错”的情况。

这正符合长度偏置的解释：错误时，长回答受到的单位惩罚更弱；模型为了降低负梯度，会把回答拉长。 所以长度增长并不一定代表推理质量提升，也可能只是优化偏差造成的 artifact(训练过程中出现的伪影 / 副作用 / 异常现象)。

# 后续改进都在针对这个机制
  - Dr. GRPO：把按真实长度归一化改成除以固定 budget，消除长度敏感性。
  - DAPO：改用全局 token 级归一化，避免长回答梯度被过度稀释。
  - GFPO：训练时多采样，再按长度和 token 效率过滤，只保留更简洁有效的回答。
  - S-GRPO / GRPO-λ：引入早退奖励或分阶段长度惩罚，鼓励“答对就尽早停止”。
  
所以，GRPO 让推理变长，并不是因为它显式奖励“长思考”，而是它的 <font color='red'>组内相对优势 + 长度归一化 + 序列级奖励广播</font> 共同制造了一个隐式的长度激励：模型学会用更多 token 来稀释错误惩罚、提高命中概率。
